## Generate Images with FLUX.1-Kontext-pro

In this notebook, we use the **FLUX.1-Kontext-pro** model from Black Forest Labs, deployed as a serverless endpoint on Azure AI Foundry, to generate images from text prompts.

FLUX.1 Kontext is a state-of-the-art image generation model known for:
- High-quality, detailed image output
- Excellent text rendering inside images
- Fast generation (~0.9s per 1024x1024 image)

We'll call the model's OpenAI-compatible `/images/generations` endpoint directly using Python.

**Prerequisite:** Deploy `FLUX.1-Kontext-pro` from the Foundry portal (Models + endpoints -> Foundry Models -> search "FLUX.1-Kontext-pro" -> Deploy). After deployment, copy the **Target URI** from the deployment details page into your `.env` file.

### Step 1: Install Packages

We need `requests` to call the REST API, `azure-identity` for authentication, and `python-dotenv` to load our settings.

In [ ]:
# Install everything we need in a single command
# - requests: makes HTTP calls to the image generation endpoint
# - azure-identity: handles Azure authentication (DefaultAzureCredential)
# - python-dotenv: loads settings from a .env file
%pip install requests azure-identity python-dotenv

### Step 2: Load Settings

Our settings are stored in a `.env` file:
- `FOUNDRY_RESOURCE_ENDPOINT` — the **Target URI** from the FLUX deployment page (e.g., `https://your-resource.cognitiveservices.azure.com`). This is NOT the project endpoint.
- `IMAGE_MODEL_DEPLOYMENT_NAME` — the FLUX deployment name (e.g., `FLUX.1-Kontext-pro`)

In [ ]:
# os lets us read environment variables from the system
import os

# base64 lets us decode the image data returned by the API
import base64

# requests lets us make HTTP calls to the FLUX endpoint
import requests

# load_dotenv reads your .env file and sets each line as an environment variable
from dotenv import load_dotenv

# DefaultAzureCredential automatically picks the best way to authenticate with Azure
from azure.identity import DefaultAzureCredential

# IPython.display lets us show images directly in a Jupyter notebook
from IPython.display import Image, display

# Load the .env file so os.getenv() can find our settings
load_dotenv()

# The Target URI from the FLUX deployment page (e.g. https://your-resource.cognitiveservices.azure.com)
# Important: this is NOT the project endpoint -- copy it from the deployment details page
my_endpoint = os.getenv("FOUNDRY_RESOURCE_ENDPOINT")

# The name of the FLUX.1-Kontext-pro deployment in your Foundry project
my_image_model = os.getenv("IMAGE_MODEL_DEPLOYMENT_NAME")

print(f"Endpoint: {my_endpoint}")
print(f"Image model: {my_image_model}")

### Step 3: Authenticate with Azure

Since API key authentication is disabled on this deployment, we use `DefaultAzureCredential` to get a **bearer token**. This automatically detects how you are logged in (Azure CLI, VS Code, managed identity, etc.).

In [ ]:
# Create a credential object that automatically detects how you are logged in
credential = DefaultAzureCredential()

# Request a bearer token scoped to Azure Cognitive Services
# This is the scope required for calling Foundry model endpoints
token = credential.get_token("https://cognitiveservices.azure.com/.default")

print("Authentication successful -- bearer token acquired.")

### Step 4: Build the API URL and Headers

FLUX.1-Kontext-pro is exposed through an OpenAI-compatible REST endpoint.  
The URL follows this pattern:

```
{target_uri}/openai/deployments/{deployment}/images/generations?api-version=2025-04-01-preview
```

We authenticate using the **bearer token** from the previous step.

In [ ]:
# The API version required for image generation with FLUX models
api_version = "2025-04-01-preview"

# Build the full URL for the image generation endpoint
generation_url = (
    f"{my_endpoint}/openai/deployments/{my_image_model}"
    f"/images/generations?api-version={api_version}"
)

# These headers are sent with every request
# - Authorization: proves who we are using the bearer token
# - Content-Type: tells the API we are sending JSON
headers = {
    "Authorization": f"Bearer {token.token}",
    "Content-Type": "application/json",
}

print(f"API URL ready: .../{my_image_model}/images/generations")

### Step 5: Generate the First Image

We send a text prompt to the FLUX model and get back a base64-encoded PNG image.  
Let's create a vintage travel poster for a fictional floating city.

In [ ]:
# The text description of the image we want to create
first_prompt = (
    "A vintage travel poster for a magical floating city above the clouds, "
    "with hot air balloons, golden sunlight, and the text "
    "'Visit Sky Harbor' at the bottom in bold retro lettering."
)

# Build the request body
# - prompt: the text description
# - n: how many images to generate (1 in our case)
# - size: the output resolution
# - output_format: the file format (png or jpeg)
request_body = {
    "prompt": first_prompt,
    "n": 1,
    "size": "1024x1024",
    "output_format": "png",
}

# Send the request to the FLUX endpoint
print("Generating image... (this may take a few seconds)")
response = requests.post(generation_url, headers=headers, json=request_body)

# Check if the request succeeded
response.raise_for_status()
result = response.json()

print("Image generated successfully!")

### Step 6: Save and Display the Image

The API returns the image as a base64-encoded string inside the `data` array.  
We decode it and save it as a PNG file, then display it in the notebook.

In [ ]:
# Extract the base64-encoded image from the response
# The response format is: {"data": [{"b64_json": "..."}]}
b64_image_data = result["data"][0]["b64_json"]

# Decode the base64 string into raw image bytes
image_bytes = base64.b64decode(b64_image_data)

# Save the image to a local file
first_filename = "sky_harbor_poster.png"
with open(first_filename, "wb") as f:
    f.write(image_bytes)

print(f"Image saved: {os.path.abspath(first_filename)}")

# Display the image inline in the notebook
display(Image(filename=first_filename))

### Step 7: Generate a Second Image (Different Prompt)

Let's try a completely different scene to see the model's range.  
This time we'll ask for a poster of a glowing underwater city.

In [ ]:
# A second prompt to test the model's versatility
second_prompt = (
    "A vintage travel poster for a glowing underwater city with coral towers, "
    "bioluminescent fish, a glass dome, and the text "
    "'Explore Coral Deep' at the bottom in art deco lettering."
)

# Build the request body for the second image
second_body = {
    "prompt": second_prompt,
    "n": 1,
    "size": "1024x1024",
    "output_format": "png",
}

# Send the request
print("Generating second image...")
second_response = requests.post(generation_url, headers=headers, json=second_body)
second_response.raise_for_status()
second_result = second_response.json()

# Decode and save
second_bytes = base64.b64decode(second_result["data"][0]["b64_json"])
second_filename = "coral_deep_poster.png"
with open(second_filename, "wb") as f:
    f.write(second_bytes)

print(f"Image saved: {os.path.abspath(second_filename)}")

# Display the image
display(Image(filename=second_filename))

### Step 8: Try a Landscape Format

FLUX.1-Kontext-pro supports different aspect ratios.  
Let's generate a wide landscape image using `1792x1024` resolution.

In [ ]:
# A wide landscape prompt
landscape_prompt = (
    "A breathtaking panoramic view of a futuristic city at sunset, "
    "with flying cars, neon-lit skyscrapers reflecting in a calm river, "
    "and cherry blossom trees lining the waterfront. Cinematic lighting."
)

# Request a wider image
landscape_body = {
    "prompt": landscape_prompt,
    "n": 1,
    "size": "1792x1024",
    "output_format": "png",
}

# Send the request
print("Generating landscape image...")
landscape_response = requests.post(generation_url, headers=headers, json=landscape_body)
landscape_response.raise_for_status()
landscape_result = landscape_response.json()

# Decode and save
landscape_bytes = base64.b64decode(landscape_result["data"][0]["b64_json"])
landscape_filename = "futuristic_city_panorama.png"
with open(landscape_filename, "wb") as f:
    f.write(landscape_bytes)

print(f"Image saved: {os.path.abspath(landscape_filename)}")

# Display the image
display(Image(filename=landscape_filename))